In [6]:
import os
print(os.getcwd())

d:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\feature_extraction


## Alternative method using donwsampled PNG by PNG for feature extraction

### Feature extraction with fivecrop function

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [8]:
from huggingface_hub import login
from transformers import AutoModel
from PIL import Image
import torch
import numpy as np
from torchvision import transforms
from typing import Union, List
import os
from conch.open_clip_custom import create_model_from_pretrained

class ConchFeatureExtractorFiveCrop:
    def __init__(self, token: str = ""):
        login(token=token)

        # Use below line to load model remotely
        # model, preprocess = create_model_from_pretrained('conch_ViT-B-16', "hf_hub:MahmoodLab/conch", hf_auth_token=token)

        # Use below line if checkpoint is saved locally
        model, preprocess = create_model_from_pretrained(
            'conch_ViT-B-16',
            r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\feature_extraction\checkpoints\CONCH\pytorch_model.bin"
        )

        # Store model and preprocess on self
        self.conch = model
        self.eval_transform = preprocess  # preprocess is the eval transform for CONCH V1

        # Add FiveCrop wrapper — self.eval_transform must be assigned before this
        self.fivecrop_transform = transforms.Compose([
            transforms.FiveCrop(256),  # yields tuple of 5 images
            transforms.Lambda(lambda crops: torch.stack([self.eval_transform(c) for c in crops]))
        ])

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.conch.to(self.device)
        self.conch.eval()

    def extract_fivecrop_features(self, image: Union[str, Image.Image]) -> torch.Tensor:
        """
        Extract 5xD features from a single image using FiveCrop.
        Returns a tensor of shape (5, D) — D=512 for CONCH V1.
        """
        if isinstance(image, str):
            if not os.path.exists(image):
                raise FileNotFoundError(f"Image not found: {image}")
            image = Image.open(image).convert("RGB")
        elif isinstance(image, Image.Image):
            image = image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image type: {type(image)}")

        crops_tensor = self.fivecrop_transform(image)  # shape: (5, C, H, W)
        crops_tensor = crops_tensor.to(self.device)

        with torch.no_grad():
            features = self.conch.encode_image(crops_tensor, proj_contrast=False, normalize=False)  # shape: (5, 512)

        return features.cpu()  # return to CPU for saving

## Test for one folder

In [7]:
import os
import torch
from PIL import Image
from tqdm import tqdm

# Assuming your ConchFeatureExtractorFiveCrop class is already defined above

def extract_features_from_png_folder(
    image_folder: str,
    output_folder: str,
    extractor: ConchFeatureExtractorFiveCrop
):
    os.makedirs(output_folder, exist_ok=True)

    png_files = [f for f in os.listdir(image_folder) if f.lower().endswith('.png')]
    print(f"🔍 Found {len(png_files)} PNG files in {image_folder}")

    for filename in tqdm(png_files, desc="Extracting features"):
        image_path = os.path.join(image_folder, filename)
        try:
            features = extractor.extract_fivecrop_features(image_path)  # (5, D)
            if features.shape[0] != 5:
                print(f"⚠️ Skipped {filename}: unexpected shape {features.shape}")
                continue

            save_path = os.path.join(output_folder, filename.replace('.png', '.pt'))
            torch.save(features.contiguous(), save_path, _use_new_zipfile_serialization=False)
        except Exception as e:
            print(f"❌ Failed on {filename}: {e}")

    print("Feature extraction complete.")


# ==== Run This Section ====
if __name__ == "__main__":
    # Instantiate extractor
    extractor = ConchFeatureExtractorFiveCrop()

    # Define input/output
    image_folder = r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data\TCGA-3L-AA1B_nonMSIH" 
    output_folder = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Test" 

    # Run extraction
    extract_features_from_png_folder(image_folder, output_folder, extractor)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful
🔍 Found 1878 PNG files in D:\Aamir Gulzar\KSA_project2\dataset\patch_data\TCGA-3L-AA1B_nonMSIH


Extracting features:   0%|          | 0/1878 [00:00<?, ?it/s]c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\layers\attention.py:80: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  x = F.scaled_dot_product_attention(
Extracting features: 100%|██████████| 1878/1878 [02:03<00:00, 15.20it/s]

Feature extraction complete.


## Final Pipeline Updated

In [9]:
from tqdm import tqdm
import pandas as pd
import torch
import os
import glob

def run_full_pipeline_fivecrop(
    patch_data_dir: str,
    extractor,
    non_white_csv: str,
    output_root: str
):
    # Load metadata
    non_white_df = pd.read_csv(non_white_csv)
    valid_patch_names = set(non_white_df['patch_name'].astype(str))

    # Iterate through all slides with tqdm
    slide_dirs = glob.glob(os.path.join(patch_data_dir, "*"))
    for slide_dir in tqdm(slide_dirs, desc="🧩 Processing slides", unit="slide", position=0):
        slide_name = os.path.basename(slide_dir)
        patch_feat_dir = os.path.join(output_root, slide_name)
        os.makedirs(patch_feat_dir, exist_ok=True)

        # Filter valid patches
        all_patches = glob.glob(os.path.join(slide_dir, "*.png"))
        valid_patch_paths = [p for p in all_patches if os.path.basename(p) in valid_patch_names]

        if not valid_patch_paths:
            tqdm.write(f"  ❌ No valid non-white patches found for {slide_name}")
            continue

        # Get already processed patch names
        existing_feats = set(os.path.splitext(f.name)[0] for f in os.scandir(patch_feat_dir) if f.name.endswith(".pt"))

        # Filter out already processed patches
        to_process = [p for p in valid_patch_paths if os.path.splitext(os.path.basename(p))[0] not in existing_feats]

        if not to_process:
            tqdm.write(f"  ⏭️ Skipped: All {len(valid_patch_paths)} patches already processed for {slide_name}")
            continue

        # ▶️ Patch-level progress bar (with position=1)
        for patch_path in tqdm(to_process, desc=f"🔄 {slide_name}", unit="patch", position=1, leave=False):
            patch_name = os.path.splitext(os.path.basename(patch_path))[0]
            save_path = os.path.join(patch_feat_dir, patch_name + ".pt")

            try:
                features = extractor.extract_fivecrop_features(patch_path)  # (5, D)
                torch.save(features, save_path)
            except Exception as e:
                tqdm.write(f"  ❌ Failed to process patch {patch_path}: {e}")
                continue

        tqdm.write(f"  ✅ Done: Processed {len(to_process)} new patches (total: {len(valid_patch_paths)}) for {slide_name}")

In [10]:
extractor = ConchFeatureExtractorFiveCrop()
import glob

run_full_pipeline_fivecrop(
    patch_data_dir=r"D:\Aamir Gulzar\KSA_project2\paip_data\patch_data",
    extractor=extractor,
    non_white_csv=r"D:\Aamir Gulzar\KSA_project2\paip_data\final_merged_without_white.csv",
    output_root=r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Conch_v1_fivecrop_Updated"
)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful


🧩 Processing slides:   1%|▏         | 1/73 [02:54<3:29:57, 174.96s/slide]

  ✅ Done: Processed 3196 new patches (total: 3196) for training_data_01_MSIH


🧩 Processing slides:   3%|▎         | 2/73 [06:46<4:06:38, 208.43s/slide]

  ✅ Done: Processed 4412 new patches (total: 4412) for training_data_02_nonMSIH


...

  ✅ Done: Processed 4171 new patches (total: 4171) for validation_data_29_MSIH


🧩 Processing slides:  99%|█████████▊| 72/73 [4:03:47<04:17, 257.98s/slide]

  ✅ Done: Processed 4118 new patches (total: 4118) for validation_data_30_nonMSIH


🧩 Processing slides: 100%|██████████| 73/73 [4:07:56<00:00, 203.79s/slide]

  ✅ Done: Processed 4810 new patches (total: 4810) for validation_data_31_nonMSIH
